# 01 - Export Llama 3.2 1B Instruct to ExecuTorch (`.pte`)  (run on Colab, CPU runtime is fine)

**Goal:** turn Meta's Llama 3.2 1B Instruct into a 4-bit-weight `.pte` file that the ExecuTorch `llama_main` runner executes on the Jetson's CPU (XNNPACK backend).
This is the competition requirement: *a Llama-family model, on-device, with ExecuTorch as the runtime*.

**Before you start (one-time, about 10 minutes)**
1. Create a free Hugging Face account: <https://huggingface.co/join>
2. Open <https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct>, read the Llama 3.2 Community License and click **Request access / Accept**. Approval is usually quick but not instant.
3. Create an access token (Settings > Access Tokens > *Read*). You will paste it in cell 2 (or store it as a Colab secret named `HF_TOKEN`).

**Why these settings?** `8da4w` = 8-bit dynamic activations + 4-bit weights (group size 32): about a quarter of the FP16 size, and the recommended recipe for the XNNPACK CPU backend.
Embeddings are quantised to 4 bits (group 32). The KV-cache and the SDPA-with-KV-cache op are enabled for fast token generation.

**Version rule:** the `executorch` version you export with and the ExecuTorch source you build `llama_main` from on the Jetson must be the **same release** (cell 5 records it).

In [ ]:

%pip -q install executorch huggingface_hub
import importlib, subprocess, sys
for pkg, imp in [("torchao", "torchao"), ("pytorch-tokenizers", "pytorch_tokenizers")]:
    try:
        importlib.import_module(imp)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
import importlib.metadata as md
print("executorch", md.version("executorch"))
for imp in ["torchao", "pytorch_tokenizers"]:
    try:
        importlib.import_module(imp); print(imp, "OK")
    except ImportError:
        print(imp, "MISSING - see the troubleshooting note at the end of this notebook")
# If pip upgraded torch, Colab may ask you to restart the runtime. Do that, then continue from cell 2.

In [ ]:

from huggingface_hub import login, snapshot_download
from pathlib import Path
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()                      # paste your token when asked

LOCAL = Path("/content/llama32_1b_instruct")
snapshot_download("meta-llama/Llama-3.2-1B-Instruct", allow_patterns=["original/*"], local_dir=str(LOCAL))
CKPT, PARAMS, TOK = LOCAL/"original/consolidated.00.pth", LOCAL/"original/params.json", LOCAL/"original/tokenizer.model"
for f in (CKPT, PARAMS, TOK):
    assert f.exists(), f"missing {f} - did you accept the license on Hugging Face?"
    print(f.name, f"{f.stat().st_size/1e6:.1f} MB")

## 3) Write the export config

`max_seq_length` is the longest prompt the model accepts; `max_context_length` is the KV-cache size (prompt + answer). Our hand-off prompt is about 350 tokens and the answer is capped at about 110 tokens, so 768 / 1024 leaves comfortable headroom while keeping RAM low.
`metadata` tells the runner which token ids end a reply (`<|eot_id|>` = 128009).

In [ ]:
EXPORT_DIR = Path("/content/llama_export"); EXPORT_DIR.mkdir(exist_ok=True)
config = f"""base:
  model_class: llama3_2
  checkpoint: {CKPT}
  params: {PARAMS}
  metadata: '{{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
quantization:
  embedding_quantize: 4,32
  qmode: 8da4w
  group_size: 32
backend:
  xnnpack:
    enabled: True
    extended_ops: True
export:
  max_seq_length: 768
  max_context_length: 1024
"""
(EXPORT_DIR / "config.yaml").write_text(config)
print(config)

## 4) Export (takes roughly 5 to 20 minutes and a lot of RAM)

If Colab runs out of memory, use *Runtime > Change runtime type > High-RAM*.
If you get an error such as *unknown key `export.max_seq_length`*, delete the `export:` block from the config in cell 3 and run again (the defaults are smaller, so then keep `--max_new_tokens` low in the advisor).

In [ ]:
import subprocess, time
t0 = time.time()
r = subprocess.run([sys.executable, "-m", "executorch.extension.llm.export.export_llm", "--config", "config.yaml"],
                   cwd=str(EXPORT_DIR), capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
print("return code", r.returncode, f"| {time.time()-t0:.0f}s")
ptes = sorted(EXPORT_DIR.glob("*.pte"), key=lambda p: p.stat().st_mtime)
assert r.returncode == 0 and ptes, "export failed - read the error above (see troubleshooting below)"
PTE = ptes[-1]
print("created", PTE, f"{PTE.stat().st_size/1e9:.2f} GB")

## 5) Save everything the Jetson needs to Drive (plus the version record)

In [ ]:
import shutil, json, datetime
from google.colab import drive
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/neu_det_yolov8n/llama_deploy"); OUT.mkdir(parents=True, exist_ok=True)
shutil.copy(PTE, OUT / "llama32_1b_instruct_8da4w.pte")
shutil.copy(TOK, OUT / "tokenizer.model")
shutil.copy(EXPORT_DIR / "config.yaml", OUT / "export_config.yaml")
ver = md.version("executorch")
info = {"created": datetime.datetime.now().isoformat(timespec="seconds"), "executorch_version": ver,
        "build_runner_from_branch": "release/" + ".".join(ver.split(".")[:2]),
        "base_model": "meta-llama/Llama-3.2-1B-Instruct", "quantization": "8da4w group 32, embeddings 4-bit group 32",
        "backend": "XNNPACK (CPU)", "max_seq_length": 768, "max_context_length": 1024,
        "pte_gb": round(PTE.stat().st_size / 1e9, 3)}
(OUT / "export_info.json").write_text(json.dumps(info, indent=2))
print(json.dumps(info, indent=2)); print("saved to", OUT)
print("\nOn the Jetson, build llama_main from:", info["build_runner_from_branch"])

## Troubleshooting and fallbacks

* **`No module named pytorch_tokenizers`**: run `git clone --branch release/1.5 https://github.com/pytorch/executorch.git` (use the branch printed above), then `pip install -e executorch/extension/llm/tokenizers/` and re-run cell 4.
* **Out of memory**: switch to a High-RAM runtime. Do not lower the quantisation unless you must.
* **Access denied when downloading**: the Hugging Face license request for Llama 3.2 is not approved yet, or the token has no read permission.
* **Alternative export route**: Hugging Face's `optimum-executorch` (`optimum-cli export executorch --model meta-llama/Llama-3.2-1B-Instruct --task text-generation --recipe xnnpack --use_custom_sdpa --use_custom_kv_cache --qlinear 8da4w --qembedding 8w`) is documented by ExecuTorch. It needs a `tokenizer.json` instead of `tokenizer.model`, so use it only if the route above fails.
* **Last-resort fallback for the deadline**: third-party pre-exported ExecuTorch builds of Llama 3.2 1B Instruct exist on Hugging Face (made for ExecuTorch 1.4). They are not from Meta, so check the license terms and the publisher before using one, and match the runner version to the file. Mention in your report that you used a pre-exported file.
* The `.pte` file is portable between machines, but the **runner you build must match the ExecuTorch version** that produced it.